# 🛡️ Banco Shield vs Hidra — Análise Exploratória de Dados

Este notebook documenta a exploração dos dados do case, cobrindo:
- Visão geral do dataset
- Qualidade dos dados e anomalias encontradas
- Análise competitiva: Shield vs Hidra
- Inadimplência e risco
- Oportunidades estratégicas

> **Dados:** 6.000 contratos financeiros fictícios — Jan a Dez/2025

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.facecolor': '#0d0d0d',
    'axes.facecolor': '#0d0d0d',
    'axes.edgecolor': '#2a2a2a',
    'axes.labelcolor': '#c8c8c8',
    'xtick.color': '#888',
    'ytick.color': '#888',
    'text.color': '#c8c8c8',
    'grid.color': '#1e1e1e',
    'grid.linestyle': '--',
    'font.family': 'monospace',
})

SC = '#4f9cf9'   # Shield
HC = '#f05252'   # Hidra
GOLD = '#f5a623'

DB = Path('..') / 'data' / 'pipeline.duckdb'
con = duckdb.connect(str(DB))
print('Conexão OK')

## 1. Visão Geral do Dataset

In [ ]:
df_raw = con.execute('SELECT * FROM bronze_fato_contratos').df()

print(f'Total de registros : {len(df_raw):,}')
print(f'Colunas            : {len(df_raw.columns)}')
print(f'Período            : {df_raw.ano_mes.min()} a {df_raw.ano_mes.max()}')
print(f'Bancos únicos      : {df_raw.bank.unique().tolist()}')
print()
df_raw.head()

In [ ]:
# Tipos e nulos
resumo = pd.DataFrame({
    'tipo': df_raw.dtypes,
    'nulos': df_raw.isna().sum(),
    'nulos_%': (df_raw.isna().mean() * 100).round(2)
})
resumo[resumo.nulos > 0]

## 2. Qualidade dos Dados

In [ ]:
qualidade = con.execute('SELECT * FROM gold_qualidade_dados').df()
qualidade

In [ ]:
# Visualização dos erros por tipo
err_cols = ['err_id_duplicado', 'err_periodo_invalido', 'err_produto_fk',
            'err_localidade_fk', 'err_valor_negativo']
err_labels = ['ID Duplicado', 'Período Inválido', 'Produto FK', 'Localidade FK', 'Valor Negativo']

shield_errs = qualidade[qualidade.bank == 'Banco Shield'][err_cols].values[0]
hidra_errs  = qualidade[qualidade.bank == 'Hidra'][err_cols].values[0]

x = range(len(err_labels))
w = 0.35

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar([i - w/2 for i in x], shield_errs, w, label='Banco Shield', color=SC)
ax.bar([i + w/2 for i in x], hidra_errs,  w, label='Hidra',        color=HC)
ax.set_xticks(list(x))
ax.set_xticklabels(err_labels)
ax.set_title('Erros por Tipo de Regra de Qualidade', pad=12)
ax.legend()
ax.grid(axis='y')
plt.tight_layout()
plt.show()

print(f"Shield: {qualidade[qualidade.bank=='Banco Shield'].pct_invalido.values[0]*100:.1f}% de registros inválidos")
print(f"Hidra : {qualidade[qualidade.bank=='Hidra'].pct_invalido.values[0]*100:.1f}% de registros inválidos")

In [ ]:
# Grafias corrigidas automaticamente
grafias = con.execute(
    "SELECT COUNT(*) as total FROM silver_fato_contratos WHERE _fixed_bank = true"
).fetchone()[0]
print(f'{grafias} registros com grafia do banco corrigida automaticamente por regex')
print('Exemplos: "hydra", "HIDRA", "Banco S.H.I.E.L.D." → normalizados para os valores canônicos')

## 3. Análise Competitiva — Shield vs Hidra

In [ ]:
carteira = con.execute('SELECT * FROM gold_carteira_banco_mes').df()
carteira['mes_label'] = carteira.ano_mes.astype(str).str[4:] + '/' + carteira.ano_mes.astype(str).str[2:4]

shield = carteira[carteira.bank == 'Banco Shield']
hidra  = carteira[carteira.bank == 'Hidra']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Saldo
axes[0].plot(shield.mes_label, shield.saldo_total / 1e6, color=SC, marker='o', label='Shield')
axes[0].plot(hidra.mes_label,  hidra.saldo_total  / 1e6, color=HC, marker='o', label='Hidra')
axes[0].set_title('Saldo da Carteira por Mês (R$ M)')
axes[0].legend()
axes[0].grid(True)
axes[0].tick_params(axis='x', rotation=45)

# Contratos
w = 0.35
x = range(len(shield))
axes[1].bar([i - w/2 for i in x], shield.contratos, w, color=SC, label='Shield')
axes[1].bar([i + w/2 for i in x], hidra.contratos,  w, color=HC, label='Hidra')
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(shield.mes_label, rotation=45)
axes[1].set_title('Novos Contratos por Mês')
axes[1].legend()
axes[1].grid(axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Share de mercado por categoria
share = con.execute('SELECT * FROM gold_share_mercado').df()

fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(share))
ax.barh([i + w/2 for i in x], share.share_shield * 100, w, color=SC, label='Shield')
ax.barh([i - w/2 for i in x], share.share_hidra  * 100, w, color=HC, label='Hidra')
ax.set_yticks(list(x))
ax.set_yticklabels(share.category)
ax.axvline(50, color='#555', linestyle='--', linewidth=1)
ax.set_xlabel('Share (%)')
ax.set_title('Share de Contratos por Categoria')
ax.legend()
ax.grid(axis='x')
plt.tight_layout()
plt.show()

seg = share[share.category == 'Seguro'].iloc[0]
print(f"⚠️  Seguro: Hidra lidera com {seg.share_hidra*100:.1f}% vs Shield {seg.share_shield*100:.1f}%")

## 4. Inadimplência e Risco

In [ ]:
df = con.execute('SELECT * FROM silver_fato_contratos WHERE _is_valid').df()

# Distribuição de DPD
df['faixa_dpd'] = pd.cut(df.dpd, bins=[-1, 0, 29, 59, 9999],
                          labels=['Adimplente', '1-29 DPD', '30-59 DPD', '60+ DPD'])

dpd_dist = df.groupby(['bank', 'faixa_dpd'], observed=True).size().unstack(fill_value=0)
dpd_pct  = dpd_dist.div(dpd_dist.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#22c55e', GOLD, HC, '#7c3aed']
dpd_pct.T.plot(kind='bar', ax=ax, color=[SC, HC], width=0.6)
ax.set_title('Distribuição de DPD por Banco (%)')
ax.set_xlabel('')
ax.set_ylabel('%')
ax.tick_params(axis='x', rotation=0)
ax.grid(axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Risk score — distribuição
fig, ax = plt.subplots(figsize=(9, 4))
for bank, color in [('Banco Shield', SC), ('Hidra', HC)]:
    ax.hist(df[df.bank == bank].risk_score, bins=30, alpha=0.6, color=color, label=bank, edgecolor='none')

ax.set_title('Distribuição do Risk Score')
ax.set_xlabel('Risk Score')
ax.set_ylabel('Frequência')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

print(df.groupby('bank')['risk_score'].describe().round(4))

In [ ]:
# Top localidades por inadimplência
inad = con.execute('SELECT * FROM gold_inadimplencia_localidade').df()
top = inad[inad.saldo_total > 0].sort_values('indice_inadimplencia', ascending=False).head(10)

fig, ax = plt.subplots(figsize=(10, 5))
colors = [SC if b == 'Banco Shield' else HC for b in top.bank]
ax.barh(top.location_name + ' (' + top.bank.str.replace('Banco Shield', 'Shield') + ')',
        top.indice_inadimplencia * 100, color=colors)
ax.set_title('Top 10 Localidades — Índice de Inadimplência (%)')
ax.set_xlabel('%')
ax.grid(axis='x')
plt.tight_layout()
plt.show()

## 5. Oportunidades Estratégicas

In [ ]:
# Nichos onde a Hidra está vulnerável
vuln = con.execute('SELECT * FROM gold_ataque_vulneravel').df()
print('🎯 Hidra Vulnerável — entrar agora com baixo risco:')
vuln[['location_name', 'product_name', 'risk_hidra', 'inad_hidra', 'score_vulnerabilidade']]

In [ ]:
# Nichos para recuperar
rec = con.execute('SELECT * FROM gold_ataque_recuperar').df()
print('🔄 Território a Recuperar — Hidra domina, Shield precisa avançar:')
rec[['location_name', 'product_name', 'share_hidra', 'share_shield', 'score_recuperacao']]

In [ ]:
# Visualização comparativa dos scores
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

label_v = vuln.location_name + ' × ' + vuln.product_name
axes[0].barh(label_v, vuln.score_vulnerabilidade, color='#22c55e')
axes[0].set_title('Score de Vulnerabilidade da Hidra')
axes[0].grid(axis='x')

label_r = rec.location_name + ' × ' + rec.product_name
axes[1].barh(label_r, rec.score_recuperacao, color=GOLD)
axes[1].set_title('Score de Recuperação de Mercado')
axes[1].grid(axis='x')

plt.tight_layout()
plt.show()

## 6. Conclusões

**Posição competitiva:**
- O Banco Shield lidera em volume de contratos e saldo de carteira em todas as categorias, exceto Seguros
- A Hidra domina Seguros com 51,8% de share — principal ameaça a monitorar

**Qualidade dos dados:**
- ~1,5% de registros inválidos em ambos os bancos — nível aceitável
- 20 grafias incorretas do nome do banco corrigidas automaticamente pelo pipeline
- 514 contratos com saldo em aberto > valor financiado — sinalizados para revisão pela área de negócio

**Risco e inadimplência:**
- Risk score médio da Hidra (~0.14) é 55% maior que o Shield (~0.09)
- Inadimplência geral controlada em ambos os bancos (~0.57%)

**Oportunidades estratégicas:**
- **Atacar:** Rio de Janeiro × Seguro Sokovia e Knowhere × Cartão Spider — Hidra com risco/inadimplência elevados
- **Recuperar:** Curitiba e Recife × Financiamento Quinjet — Hidra com 76-86% de share, Shield sub-representado

In [ ]:
con.close()